# Solana Next-Day High Prediction

This notebook trains and evaluates anchored residual models for predicting the next daily high price of Solana (SOL/USD). It now uses Kraken's public OHLC endpoint as the primary data source, so the workflow does not depend on local raw CSV files.

The production code lives in `solana_price_prediction/`; this notebook is the readable experiment layer over the same functions used by the API and Streamlit dashboard.

## 1. Setup

The notebook imports reusable project modules for Kraken data access, cleaning, feature engineering, time-series splitting, training, and inference. This keeps notebook logic aligned with production code.

In [ ]:
from pathlib import Path
import sys

# Make package imports work whether this notebook is run from the repo root,
# the notebooks/ folder, or JupyterLab's configured working directory.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "solana_price_prediction").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root containing 'solana_price_prediction/'. "
        "Run this notebook from inside the cloned repository."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from solana_price_prediction.config import MODELS_DIR, PROCESSED_DATA_DIR
from solana_price_prediction.dataset import clean_solana_data
from solana_price_prediction.features import (
    EXPECTED_FEATURES,
    SolanaFeatureEngineer,
    add_features,
    create_target_variable,
)
from solana_price_prediction.kraken import fetch_kraken_ohlc, kraken_since_timestamp
from solana_price_prediction.modeling.estimators import DeltaHighRegressor, PersistenceHighRegressor
from solana_price_prediction.modeling.predict import predict_next_high
from solana_price_prediction.modeling.train import evaluate_baseline, split_time_series

pd.set_option("display.max_columns", None)

PAIR = "SOLUSD"
INTERVAL_MINUTES = 1440
LOOKBACK_DAYS = 720
MODEL_PATH = MODELS_DIR / "solana_next_day_high.joblib"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model output: {MODEL_PATH}")

## 2. Fetch Market Data From Kraken

Kraken endpoint used by the project:

```text
https://api.kraken.com/0/public/OHLC?pair=SOLUSD&interval=1440&since=<unix_timestamp>
```

Daily candles are enough for the current target: next-day high price. `marketcap` is set to `0.0` because Kraken OHLC does not provide market capitalization.

In [ ]:
since_timestamp = kraken_since_timestamp(LOOKBACK_DAYS)
raw_df = fetch_kraken_ohlc(
    pair=PAIR,
    interval=INTERVAL_MINUTES,
    since_timestamp=since_timestamp,
)

raw_df.head()

In [ ]:
raw_df.info()
raw_df.tail()

## 3. Clean And Prepare The Modeling Dataset

The model predicts `target_next_day_high`, created by shifting the daily `high` column one row into the future. Feature generation adds moving averages, RSI, volatility, and lagged high/volume features.

In [ ]:
clean_df = clean_solana_data(raw_df)
model_df = create_target_variable(clean_df)
feature_df = add_features(model_df)

feature_df.to_parquet(PROCESSED_DATA_DIR / "solana_features.parquet", index=False)

print(f"Raw rows: {len(raw_df):,}")
print(f"Feature rows: {len(feature_df):,}")
feature_df[["timestamp", "high", "target_next_day_high", "sma_7", "sma_30", "rsi_14"]].tail()

## 4. Chronological Train / Validation / Test Split

For time-series data, rows must stay in date order. The default split is:

- 70% train
- 15% validation for early stopping
- 15% test for final reporting

The model never trains on future rows relative to the validation or test windows.

In [ ]:
train_df, validation_df, test_df = split_time_series(
    feature_df,
    validation_size=0.15,
    test_size=0.15,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "start": [train_df["timestamp"].min(), validation_df["timestamp"].min(), test_df["timestamp"].min()],
        "end": [train_df["timestamp"].max(), validation_df["timestamp"].max(), test_df["timestamp"].max()],
    }
)
split_summary

## 5. Model Training Workbench

The previous plain XGBoost model was weak because tree models struggle when asked to extrapolate absolute crypto price levels. This workbench compares several production-compatible strategies:

- Persistence baseline: tomorrow's high equals today's high
- Direct models: predict next-day high directly
- Residual models: predict `next_day_high - current_high`, then add current high back

The validation window selects the model. The test window is used once for final reporting.

In [ ]:
FEATURES = EXPECTED_FEATURES
TARGET = "target_next_day_high"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]
X_test = test_df[FEATURES]
y_test = test_df[TARGET]

train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)
X_train_validation = train_validation_df[FEATURES]
y_train_validation = train_validation_df[TARGET]

print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Test rows: {len(X_test):,}")

## 6. Candidate Models

Direct tree models are intentionally excluded from selection because they predict absolute price levels and can fail badly during regime shifts. Every production candidate is anchored to today's high, either as the persistence baseline or as a residual model that predicts `next_day_high - current_high`.

A validation guardrail is applied: ML is selected only if it beats persistence by at least 2% on validation MAE. Otherwise, the saved joblib uses the persistence baseline, which is often hard to beat for next-day OHLC targets.

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred),
    }


candidate_models = {
    "persistence_high": PersistenceHighRegressor(),
    "ridge_delta_high": DeltaHighRegressor(
        make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        residual_clip=20.0,
    ),
    "random_forest_delta_high": DeltaHighRegressor(
        RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1,
        ),
        residual_clip=20.0,
    ),
    "hist_gradient_boosting_delta_high": DeltaHighRegressor(
        HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.03,
            max_leaf_nodes=10,
            min_samples_leaf=20,
            l2_regularization=5.0,
            random_state=42,
        ),
        residual_clip=20.0,
    ),
    "xgboost_delta_high": DeltaHighRegressor(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.02,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=2.0,
            reg_lambda=5.0,
            random_state=42,
            n_jobs=-1,
        ),
        residual_clip=20.0,
    ),
}

validation_rows = []
fitted_candidates = {}

for name, candidate in candidate_models.items():
    fitted = clone(candidate).fit(X_train, y_train)
    fitted_candidates[name] = fitted
    validation_pred = fitted.predict(X_validation)
    validation_rows.append({"model": name, **regression_metrics(y_validation, validation_pred)})

validation_report = pd.DataFrame(validation_rows).sort_values("mae")
persistence_validation_mae = validation_report.loc[
    validation_report["model"] == "persistence_high", "mae"
].iloc[0]
validation_report["mae_vs_persistence"] = validation_report["mae"] - persistence_validation_mae
validation_report

## 7. Select, Refit, And Test

The best validation model is refit on train + validation before final testing. This gives the selected strategy the maximum historical data available before the test window.

In [ ]:
MIN_VALIDATION_IMPROVEMENT = 0.02

best_row = validation_report.iloc[0]
if best_row["model"] != "persistence_high" and best_row["mae"] < persistence_validation_mae * (1 - MIN_VALIDATION_IMPROVEMENT):
    selected_model_name = best_row["model"]
else:
    selected_model_name = "persistence_high"

selected_template = candidate_models[selected_model_name]

model = clone(selected_template).fit(X_train_validation, y_train_validation)
test_predictions = model.predict(X_test)

test_metrics = regression_metrics(y_test, test_predictions)
baseline_test_pred = PersistenceHighRegressor().fit(X_train_validation, y_train_validation).predict(X_test)
baseline_test_metrics = regression_metrics(y_test, baseline_test_pred)

results_df = test_df.copy()
results_df["predicted_high"] = test_predictions
results_df["error"] = results_df[TARGET] - results_df["predicted_high"]

print(f"Selected model: {selected_model_name}")
print(f"Validation persistence MAE: ${persistence_validation_mae:,.4f}")
print("Selected model test metrics")
print(f"MAE:  ${test_metrics['mae']:,.4f}")
print(f"RMSE: ${test_metrics['rmse']:,.4f}")
print(f"R2:    {test_metrics['r2']:,.4f}")
print("Persistence test metrics")
print(f"MAE:  ${baseline_test_metrics['mae']:,.4f}")
print(f"RMSE: ${baseline_test_metrics['rmse']:,.4f}")
print(f"R2:    {baseline_test_metrics['r2']:,.4f}")

results_df[["timestamp", TARGET, "high", "predicted_high", "error"]].tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(results_df["timestamp"], results_df[TARGET], label="Actual next-day high", alpha=0.8)
ax.plot(results_df["timestamp"], results_df["predicted_high"], label=f"Predicted ({selected_model_name})", linestyle="--")
ax.plot(results_df["timestamp"], results_df["high"], label="Persistence baseline", alpha=0.5)
ax.set_title("Solana Next-Day High: Actual vs Predicted")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 8. Model Inspection

Feature importance is available for tree-based models. For wrapped residual models, inspect the inner estimator.

In [ ]:
def unwrap_estimator(fitted_model):
    return getattr(fitted_model, "estimator_", fitted_model)

inner_model = unwrap_estimator(model)

if hasattr(inner_model, "feature_importances_"):
    feature_names = list(getattr(inner_model, "feature_names_in_", FEATURES))
    feature_importance = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": inner_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
else:
    feature_importance = pd.DataFrame(
        {"message": [f"{selected_model_name} does not expose feature_importances_."]}
    )

feature_importance

## 9. Save Fresh Joblib

The saved object must expose `.predict(X)` and return next-day high prices. The API and batch inference commands can load this joblib directly.

In [ ]:
joblib.dump(model, MODEL_PATH)
results_df.to_parquet(PROCESSED_DATA_DIR / "solana_predictions.parquet", index=False)

loaded_model = joblib.load(MODEL_PATH)
loaded_prediction = predict_next_high(loaded_model, clean_df)

print(f"Saved selected model to {MODEL_PATH}")
print(f"Saved model type: {type(loaded_model).__name__}")
print(f"Latest prediction smoke test: ${loaded_prediction:,.2f}")

## 10. Streamlit Dashboard Integration

The Streamlit dashboard in `dashboard/app/` uses the same Kraken source for live candles and calls the FastAPI service for model predictions.

After saving a new model, copy it to the API model directory if you want the local API to use it:

```powershell
Copy-Item models/solana_next_day_high.joblib api/models/solana_next_day_high.joblib -Force
```

Run the API first:

```powershell
cd api
uvicorn app.main:app --reload --port 8000
```

Then run the dashboard:

```powershell
cd dashboard
streamlit run app/main.py
```

In [ ]:
# Minimal Streamlit pattern used by the dashboard.
# Keep this as a reference snippet; run the real app from dashboard/app/main.py.

STREAMLIT_SNIPPET = """
import streamlit as st
from utils import fetch_kraken_data

st.title("Solana Intelligence")
df = fetch_kraken_data("SOLUSD", interval=1440)
st.line_chart(df.set_index("date")[["close"]])
"""

print(STREAMLIT_SNIPPET)